# Projet Home Credit — Sujet 03 · Séance 01

**Membre :** Wafik A. (`ADKfull-dev`)
**Porteur :** P3

**Étapes assignées :**
- 1.4 : cardinalité
- 1.6 : trous (valeurs manquantes)
- 1.9 : cible

**Objectif du sujet :** décider d'accorder ou de refuser un crédit. Ce carnet ne modélise pas : il
décrit les données pour préparer les décisions des étapes suivantes.

## 0. Trouver et charger les données

Les données ne sont pas dans le dépôt (trop lourdes). Le carnet les cherche tout seul :

- **en local** : dans un dossier `Sujet_03_Home_Credit/` placé à côté du carnet ou à la racine du dépôt,
  ou dans le dossier indiqué par la variable d'environnement `DONNEES_PROJET` ;
- **sur Kaggle** : n'importe où sous `/kaggle/input/` (après **Add Input**) ;
- **sur Colab** : dans `/content/` ou `MyDrive/Sujet_03_Home_Credit/`.

In [1]:
from pathlib import Path
import os
import pandas as pd

FICHIERS = ["train.csv", "a_decider.csv"]
SUJET = "Sujet_03_Home_Credit"


def trouver_donnees():
    """Renvoie le premier dossier qui contient tous les FICHIERS."""
    ici = Path.cwd().resolve()
    pistes = []
    if os.environ.get("DONNEES_PROJET"):
        pistes.append(Path(os.environ["DONNEES_PROJET"]))
    for dossier in [ici, *list(ici.parents)[:3]]:
        pistes += [dossier, dossier / SUJET]
    pistes += [Path("/content"), Path("/content/drive/MyDrive") / SUJET]
    for piste in pistes:
        if all((piste / f).is_file() for f in FICHIERS):
            return piste
    # Kaggle : le jeu de données peut être rangé plus profondément
    if Path("/kaggle/input").exists():
        for chemin in Path("/kaggle/input").rglob("train.csv"):
            if (chemin.parent / "a_decider.csv").is_file():
                return chemin.parent
    raise FileNotFoundError(
        "Données introuvables : placez le dossier Sujet_03_Home_Credit/ à côté du carnet "
        "ou définissez DONNEES_PROJET=/chemin/vers/le/dossier."
    )


DOSSIER = trouver_donnees()
train = pd.read_csv(DOSSIER / "train.csv")
a_decider = pd.read_csv(DOSSIER / "a_decider.csv")
print("dossier de données :", DOSSIER.name)
print("train :", train.shape, "| a_decider :", a_decider.shape)

dossier de données : Sujet_03_Home_Credit
train : (246008, 122) | a_decider : (61503, 121)


## 1.4 — Cardinalité : combien de valeurs différentes par colonne ?

La cardinalité indique **comment traiter chaque colonne** : une colonne à une seule valeur
n'apprend rien au modèle, une colonne à 2 valeurs est un oui/non, et une colonne texte avec beaucoup
de modalités devra être encodée avec soin.

In [2]:
cardinalite = train.nunique().sort_values()   # nunique ignore les valeurs manquantes

print("Colonnes avec 1 seule valeur :", (cardinalite == 1).sum(), "->", list(cardinalite[cardinalite == 1].index))
deux = cardinalite[cardinalite == 2].index
print("Colonnes avec 2 valeurs (oui/non) :", len(deux), "(dont la cible TARGET)" if "TARGET" in deux else "")

texte = [c for c in train.columns if not pd.api.types.is_numeric_dtype(train[c])]   # les colonnes en texte (catégories)
cardinalite[texte].sort_values(ascending=False)

Colonnes avec 1 seule valeur : 1 -> ['FLAG_MOBIL']
Colonnes avec 2 valeurs (oui/non) : 36 (dont la cible TARGET)


ORGANIZATION_TYPE             58
OCCUPATION_TYPE               18
NAME_INCOME_TYPE               8
NAME_TYPE_SUITE                7
WEEKDAY_APPR_PROCESS_START     7
WALLSMATERIAL_MODE             7
NAME_FAMILY_STATUS             6
NAME_HOUSING_TYPE              6
NAME_EDUCATION_TYPE            5
FONDKAPREMONT_MODE             4
CODE_GENDER                    3
HOUSETYPE_MODE                 3
NAME_CONTRACT_TYPE             2
FLAG_OWN_CAR                   2
FLAG_OWN_REALTY                2
EMERGENCYSTATE_MODE            2
dtype: int64

In [3]:
# CODE_GENDER a 3 valeurs alors qu'on en attend 2 : regardons pourquoi
train["CODE_GENDER"].value_counts(dropna=False)

CODE_GENDER
F      162113
M       83892
XNA         3
Name: count, dtype: int64

**Ce qu'on retient :**

- **`FLAG_MOBIL` est constante** (tout le monde a fourni un mobile) : elle n'apporte aucune
  information et peut être retirée.
- Parmi les 36 colonnes à 2 valeurs, il y a **`TARGET` elle-même** : il reste donc 35 variables
  oui/non.
- **`CODE_GENDER` a 3 valeurs** parce que 3 demandes portent la valeur `XNA` (non renseigné) : c'est
  en réalité un manquant déguisé.
- **`ORGANIZATION_TYPE` a 58 modalités** : un encodage « une colonne par modalité » crée beaucoup de
  colonnes ; il faudra peut-être regrouper les modalités rares.

## 1.6 — Trous : où sont les valeurs manquantes ?

L'énoncé prévient que les manquants **n'ont pas été comblés** et qu'**ils sont une information**.
On mesure donc leur ampleur, on vérifie que `a_decider.csv` a le même profil, puis on regarde si le
fait qu'une valeur manque est lié au défaut.

In [4]:
trous = train.isna().mean() * 100        # % de manquants par colonne
trous_a_decider = a_decider.isna().mean() * 100

print("Colonnes avec des trous :", (trous > 0).sum())
print("Colonnes avec plus de 30 % de trous :", (trous > 30).sum())
print("Le pire :", round(trous.max()), "% de trous")
print("a_decider, colonnes > 30 % :", (trous_a_decider > 30).sum(), "| le pire :", round(trous_a_decider.max()), "%")

trous[trous > 0].sort_values(ascending=False).head(10).round(1)

Colonnes avec des trous : 66
Colonnes avec plus de 30 % de trous : 50
Le pire : 70 % de trous
a_decider, colonnes > 30 % : 50 | le pire : 70 %


COMMONAREA_MEDI             69.9
COMMONAREA_AVG              69.9
COMMONAREA_MODE             69.9
NONLIVINGAPARTMENTS_MEDI    69.5
NONLIVINGAPARTMENTS_MODE    69.5
NONLIVINGAPARTMENTS_AVG     69.5
LIVINGAPARTMENTS_MODE       68.4
LIVINGAPARTMENTS_MEDI       68.4
LIVINGAPARTMENTS_AVG        68.4
FONDKAPREMONT_MODE          68.4
dtype: float64

In [5]:
# Le fait qu'une valeur manque change-t-il le taux de défaut ?
colonnes = ["EXT_SOURCE_1", "EXT_SOURCE_3", "OWN_CAR_AGE", "OCCUPATION_TYPE"]
lignes = []
for col in colonnes:
    manque = train[col].isna()
    lignes.append({
        "colonne": col,
        "% manquants": round(manque.mean() * 100, 1),
        "défaut si manquant (%)": round(train.loc[manque, "TARGET"].mean() * 100, 2),
        "défaut si présent (%)": round(train.loc[~manque, "TARGET"].mean() * 100, 2),
    })
pd.DataFrame(lignes)

,colonne,% manquants,défaut si manquant (%),défaut si présent (%)
0,EXT_SOURCE_1,56.4,8.51,7.50
1,EXT_SOURCE_3,19.9,9.33,7.76
2,OWN_CAR_AGE,66.0,8.48,7.28
3,OCCUPATION_TYPE,31.3,6.51,8.79


**Ce qu'on retient :**

- **50 colonnes dépassent 30 % de manquants**, jusqu'à 70 % : c'est exactement le chiffre de
  l'énoncé, et `a_decider.csv` a le même profil. Ce qu'on décidera sur `train.csv` s'appliquera donc
  aussi aux demandes à décider.
- **Les manquants sont liés au défaut**, dans un sens ou dans l'autre : quand `EXT_SOURCE_3` manque,
  le défaut monte à 9,3 % (contre 7,8 %) ; quand `OCCUPATION_TYPE` manque, il descend à 6,5 %
  (contre 8,8 %). Supprimer ces colonnes ou les combler sans le signaler ferait perdre cette
  information ; une piste est d'ajouter un indicateur « valeur manquante » à côté de la colonne.

## 1.9 — La cible : combien de clients en défaut ?

In [6]:
print(train["TARGET"].value_counts())   # nombre de 0 (remboursé) et de 1 (défaut)

taux = train["TARGET"].mean() * 100     # % de défauts
print("Taux de défaut :", round(taux, 2), "%")
print("Valeurs manquantes dans TARGET :", train["TARGET"].isna().sum())

TARGET
0    226148
1     19860
Name: count, dtype: int64
Taux de défaut : 8.07 %
Valeurs manquantes dans TARGET : 0


**Ce qu'on retient :**

- **8,07 % de défauts** (19 860 sur 246 008), comme dans l'énoncé : les classes sont très
  déséquilibrées.
- Un modèle qui accorde **tous** les crédits aurait déjà **91,9 % d'exactitude** sans rien détecter :
  l'exactitude (*accuracy*) n'est donc pas une bonne métrique ici. Il faudra une métrique qui tient
  compte du coût des erreurs (bloc 1).
- Pour valider, il faudra garder cette proportion de défauts dans chaque partie du découpage
  (découpage **stratifié**).